## Get Model PTQ Quantization

In [ ]:
pip install torch
import copy
import torch
import torch.nn.functional as F
from ptq_utils import is_target, load_models

In [12]:
# Tensor quantization
import torch

x = torch.randn(10)
s = (x.max() - x.min()) / 255 # scale
zp = torch.round(-x.min() / s)  # representation of floating 0.0 in the quantized scale
x_q = torch.clamp(torch.round(x / s) + zp, 0, 255) 
x_hat = (x_q - zp) * s # dequantize

print("Tensor:", x)
print("Scale:", s)
print("Zero:", zp)
print("Quantized x:", x_q)
print("Dequantized x:", x_hat)
print("Error", (x - x_hat).abs().max(), s / 2) # Error <= s / 2 is tolerable



# Shape: (out_channels, in_channels, height, width)
w = torch.randn(16, 3, 5, 5)

# These two lines do the exact same thing:
scale_dim  = w.abs().amax(dim=[1, 2, 3]) / 127
scale_axis = w.abs().amax(axis=[1, 2, 3]) / 127

# This is also the same as it flattens the vector from dimension 1 forward and find the max of the newly collapsed column
# example 16 x 3 x 3 x 3 = 16 x 9, and find max of dimension 9 (columns)
scale_dim = w.abs().flatten(1).amax(dim=1) / 127 

print(scale_dim)
print(scale_axis)
# Both result in a flat tensor of 16 channel scales
print(torch.equal(scale_dim, scale_axis))  # True



Tensor: tensor([-1.2428, -0.3277, -0.6021,  0.0684,  1.5674,  2.1196, -1.1530, -0.6455,
        -0.9537,  1.0434])
Scale: tensor(0.0132)
Zero: tensor(94.)
Quantized x: tensor([  0.,  69.,  48.,  99., 213., 255.,   7.,  45.,  22., 173.])
Dequantized x: tensor([-1.2395, -0.3296, -0.6065,  0.0659,  1.5691,  2.1229, -1.1472, -0.6461,
        -0.9494,  1.0417])
Error tensor(0.0058) tensor(0.0066)
tensor([0.0165, 0.0199, 0.0250, 0.0197, 0.0209, 0.0208, 0.0191, 0.0192, 0.0206,
        0.0180, 0.0177, 0.0171, 0.0308, 0.0166, 0.0183, 0.0200])
tensor([0.0165, 0.0199, 0.0250, 0.0197, 0.0209, 0.0208, 0.0191, 0.0192, 0.0206,
        0.0180, 0.0177, 0.0171, 0.0308, 0.0166, 0.0183, 0.0200])
True


In [ ]:
# Per output channel scale
# Symmetric vs Asymmetric quantization
# Symmetric keeps the center around 0.0
# - use amax()
# Asymmetric maps the center from the 0.0 to a number inside the floating range
# - use standard affine transform formula


# s = w.abs().amax(dim=(1,2,3), keep_dim=True) / 127

# Functional 2D Convolution
# F.conv2d(input, weight=filters, bias, stride, padding, dilation, groups)

# Normal 2D Convolution
# nn.conv2d(in_channel, out_channel, kernel_size, ...)

def fake_quantize_weight(w, per_channel):
    """ 
    Symmetric quantization
    Conv2d weight = [Cout, Cin, kH, kW]
    Return dequantized weight
    """
    if per_channel:
        # Find symmetric scale
        # keepdim=True -> s has shape [Cout,1,1,1] so it broadcasts against w [Cout,Cin,kH,kW]
        s = w.abs().amax(dim=(1,2,3), keepdim=True).clamp(min=1e-8) / 127
    else:
        # per-tensor: one scalar scale for the whole weight (still symmetric, zero_point = 0)
        s = w.abs().amax().clamp(min=1e-8) / 127

    # symmetric: no zero point
    w_q = torch.clamp(torch.round(w / s), -127, 127)
    # Dequantize
    w_hat = w_q * s

    # every element's error must be within half a quantization step
    assert ((w - w_hat).abs() <= s / 2 + 1e-6).all()
    return w_hat

class QuantConv2d(torch.nn.Module):
    def __init__(self, conv, per_channel=True) -> None:
        super().__init__() # Call PyTorch's instantiation method
        # buffer = saved with state_dict and moved by .to(device), but not trained
        self.register_buffer("w_hat", fake_quantize_weight(conv.weight.detach(), per_channel))
        bias = conv.bias.detach().clone() if conv.bias is not None else None
        self.register_buffer("bias", bias)   # None is allowed
        # these are tuples like (1,1), never None
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.groups = conv.groups

    def forward(self, x):
        # order: input, weight, bias, stride, padding, dilation, groups
        return torch.F.conv2d(x, self.w_hat, self.bias, self.stride, self.padding, self.dilation, self.groups)


def _set_submodule(root, dotted_name, new_module):
    """Replace root.<a>.<b>.<c> given 'a.b.c'."""
    *parent_path, last = dotted_name.split(".")
    parent = root
    for p in parent_path:
        parent = getattr(parent, p)
    setattr(parent, last, new_module)


def swap_model(model, per_channel=True):
    model_q = copy.deepcopy(model)   # keep the FP32 baseline untouched
    # collect first, swap after: don't modify the module tree while iterating it
    targets = [(n, m) for n, m in model_q.named_modules() if is_target(n, m)]
    for name, m in targets:
        _set_submodule(model_q, name, QuantConv2d(m, per_channel))
    return model_q


# Self-test: QuantConv2d should closely match nn.Conv2d
conv = torch.nn.Conv2d(3, 8, 3, padding=1)
qconv = QuantConv2d(conv, per_channel=True)
xt = torch.randn(1, 3, 16, 16)
print("max abs diff:", (conv(xt) - qconv(xt)).abs().max())



NameError: name 'F' is not defined

In [ ]:
# # TODO Stage A (finish): evaluate weight-only fake quant
# # [ ] vae, mapper, scale, cfg = load_models(ROOT, device=DEVICE)
# ROOT = "/content/drive/MyDrive/CSCE-491"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# load_models(ROOT, )
# [ ] calib_loader, test_loader = get_dataloaders(cfg["resolved_pickle_path"], n_calib=64, n_eval=50)
# [ ] base = evaluate(vae, mapper, test_loader, scale, DEVICE)                 # FP32 baseline
# [ ] swap mapper only first (tiny), then VAE, then both:
#       mapper_q = swap_model(mapper, per_channel=True)
#       vae_q    = swap_model(vae,    per_channel=True)
# [ ] compare per_channel=True vs False -> expect per-channel to be closer to baseline
# [ ] print estimate_int8_size_mb(vae) vs get_state_dict_size_mb(vae)
# [ ] log a results row: device, config, DSC, IoU, HD95, ms/img, MB

In [ ]:
# TODO Stage B: simulated W8A8 (activation fake quant + calibration)  [Colab T4]
# [ ] Extend QuantConv2d: quant_act flag, calibrating flag, x_min / x_max buffers
#       (init torch.tensor(float('inf')) / torch.tensor(float('-inf')))
# [ ] forward, calibrating: update x_min/x_max with torch.minimum/maximum, run plain conv
# [ ] finalize(): include 0 in range (min(x_min,0), max(x_max,0)),
#       scale = (x_max - x_min)/255 ; zp = round(-x_min/scale) clamped to [0,255]
# [ ] forward, quant_act: x = (clamp(round(x/scale)+zp, 0, 255) - zp) * scale before conv
# [ ] calibrate(vae, mapper, calib_loader): set flag, loop forward_pipeline under no_grad, finalize all
# [ ] compare range estimators: min/max vs percentile (subsample before torch.quantile) vs MSE clip
# [ ] sensitivity sweep: swap only one get_layer_group(...) at a time, rank by DSC drop
# [ ] decide final SKIP set; save skip list + per-layer ranges to Drive (calib.pt)

In [ ]:
# TODO Stage C (TensorRT INT8 on T4, Colab)
# Note: torch.ao INT8 is CPU-only; TensorRT is what uses the T4's INT8 Tensor Cores.
# [ ] check versions: nvidia-smi (CUDA), pip install tensorrt onnx nvidia-modelopt
# [ ] insert Q/DQ nodes with ModelOpt using the Stage B calibration data:
#       import modelopt.torch.quantization as mtq
#       model = mtq.quantize(model, mtq.INT8_DEFAULT_CFG, forward_loop=calib_loop)
# [ ] disable quantizers on SKIP layers (boundary convs, depthwise, GroupNorm/SiLU/softmax stay FP)
# [ ] export vae.encoder, mapper, vae.decoder to ONNX separately (opset >= 17, fixed 1x224x224 shapes)
# [ ] build engines: trtexec --onnx=X.onnx --int8 --fp16 --saveEngine=X.plan
#       also build FP16-only engines as the fair TensorRT baseline
# [ ] run the pipeline with TRT engines in place of the PyTorch modules; reuse evaluate() metrics
# [ ] benchmark ms/img: PyTorch FP32 CUDA vs TRT FP16 vs TRT INT8 (warm-up, cuda sync, fixed batch=1)
# [ ] check accuracy: TRT INT8 DSC should be close to Stage B simulated W8A8
# [ ] optional: Mac CPU real INT8 with torch.ao + qnnpack (QWrap/prepare/convert) for comparison

In [ ]:
# TODO Report
# [ ] one results table: device | config | DSC | IoU | HD95 | ms/img | MB
# [ ] rows: FP32, A per-tensor, A per-channel, B (best estimator), TRT FP16, TRT INT8
# [ ] update Quantization_Methodology.md keep/quantize list with sweep findings